# 📦 Dataset Trimmer — Emotion Chatbot Project
Trims 3 datasets to under 2,000 rows each, with stratified sampling to keep class balance.

**Datasets:**
- EmpatheticDialogues (Facebook AI)
- Spotify Lyrics Dataset
- GoEmotions (Google)

In [ ]:
import pandas as pd
import numpy as np
import os
import glob

# Target row count
TARGET_ROWS = 1800  # safely below 2000
RANDOM_SEED = 42

print('Libraries loaded ✅')
print(f'Target rows per dataset: {TARGET_ROWS}')

---
## 🔍 Step 1 — Explore Available Files

In [ ]:
base_path = '/kaggle/input'

# List all CSV files across all datasets
all_csv = glob.glob(f'{base_path}/**/*.csv', recursive=True)
all_tsv = glob.glob(f'{base_path}/**/*.tsv', recursive=True)

print('=== CSV files found ===')
for f in all_csv:
    print(f)

print('\n=== TSV files found ===')
for f in all_tsv:
    print(f)

---
## 📘 Dataset 1 — EmpatheticDialogues (Facebook AI)

In [ ]:
# --- Load EmpatheticDialogues ---
empathetic_path = '/kaggle/input/datasets/atharvjairath/empathetic-dialogues-facebook-ai'

# Check what files exist
print('Files in EmpatheticDialogues folder:')
for f in glob.glob(f'{empathetic_path}/**/*', recursive=True):
    print(f)

In [ ]:
# Load the CSV (adjust filename if needed based on output above)
emp_files = glob.glob(f'{empathetic_path}/**/*.csv', recursive=True)
if not emp_files:
    emp_files = glob.glob(f'{empathetic_path}/**/*.tsv', recursive=True)

print(f'Found: {emp_files}')

# Load first file found
try:
    df_emp = pd.read_csv(emp_files[0])
except Exception:
    df_emp = pd.read_csv(emp_files[0], sep='\t')

print(f'\nOriginal shape: {df_emp.shape}')
print(f'Columns: {list(df_emp.columns)}')
df_emp.head(3)

In [ ]:
# --- Trim EmpatheticDialogues ---
# Detect emotion column (usually 'context' or 'emotion')
emotion_col_emp = None
for candidate in ['context', 'emotion', 'label', 'situation']:
    if candidate in df_emp.columns:
        emotion_col_emp = candidate
        break

print(f'Emotion column detected: {emotion_col_emp}')

if emotion_col_emp and df_emp[emotion_col_emp].nunique() > 1:
    # Stratified sampling — keep class balance
    n_classes = df_emp[emotion_col_emp].nunique()
    per_class = TARGET_ROWS // n_classes
    print(f'Classes: {n_classes} | Rows per class: {per_class}')

    df_emp_trimmed = (
        df_emp.groupby(emotion_col_emp, group_keys=False)
        .apply(lambda x: x.sample(min(len(x), per_class), random_state=RANDOM_SEED))
        .reset_index(drop=True)
    )
else:
    # Random sampling fallback
    df_emp_trimmed = df_emp.sample(n=min(TARGET_ROWS, len(df_emp)), random_state=RANDOM_SEED).reset_index(drop=True)

print(f'\nOriginal rows : {len(df_emp)}')
print(f'Trimmed rows  : {len(df_emp_trimmed)}')
df_emp_trimmed.head(3)

---
## 🎵 Dataset 2 — Spotify Lyrics Dataset

In [ ]:
# --- Load Spotify Lyrics ---
spotify_path = '/kaggle/input/datasets/evabot/spotify-lyrics-dataset'

print('Files in Spotify folder:')
for f in glob.glob(f'{spotify_path}/**/*', recursive=True):
    print(f)

In [ ]:
spot_files = glob.glob(f'{spotify_path}/**/*.csv', recursive=True)
print(f'Found: {spot_files}')

df_spot = pd.read_csv(spot_files[0])

print(f'\nOriginal shape: {df_spot.shape}')
print(f'Columns: {list(df_spot.columns)}')
df_spot.head(3)

In [ ]:
# --- Trim Spotify Lyrics ---
# Detect mood/genre column for stratified sampling
mood_col_spot = None
for candidate in ['mood', 'genre', 'tag', 'label', 'category']:
    if candidate in df_spot.columns:
        mood_col_spot = candidate
        break

print(f'Mood/genre column detected: {mood_col_spot}')

if mood_col_spot and df_spot[mood_col_spot].nunique() > 1:
    n_classes = df_spot[mood_col_spot].nunique()
    per_class = TARGET_ROWS // n_classes
    print(f'Classes: {n_classes} | Rows per class: {per_class}')

    df_spot_trimmed = (
        df_spot.groupby(mood_col_spot, group_keys=False)
        .apply(lambda x: x.sample(min(len(x), per_class), random_state=RANDOM_SEED))
        .reset_index(drop=True)
    )
else:
    df_spot_trimmed = df_spot.sample(n=min(TARGET_ROWS, len(df_spot)), random_state=RANDOM_SEED).reset_index(drop=True)

# Drop rows with missing lyrics if column exists
for col in ['lyrics', 'lyric', 'text']:
    if col in df_spot_trimmed.columns:
        before = len(df_spot_trimmed)
        df_spot_trimmed = df_spot_trimmed.dropna(subset=[col]).reset_index(drop=True)
        print(f'Dropped {before - len(df_spot_trimmed)} rows with missing {col}')
        break

print(f'\nOriginal rows : {len(df_spot)}')
print(f'Trimmed rows  : {len(df_spot_trimmed)}')
df_spot_trimmed.head(3)

---
## 😊 Dataset 3 — GoEmotions (Google)

In [ ]:
# --- Load GoEmotions ---
goemo_path = '/kaggle/input/datasets/debarshichanda/goemotions'

print('Files in GoEmotions folder:')
for f in glob.glob(f'{goemo_path}/**/*', recursive=True):
    print(f)

In [ ]:
goemo_files = glob.glob(f'{goemo_path}/**/*.csv', recursive=True)
if not goemo_files:
    goemo_files = glob.glob(f'{goemo_path}/**/*.tsv', recursive=True)

print(f'Found: {goemo_files}')

# GoEmotions sometimes splits into train/dev/test — merge all
dfs = []
for f in goemo_files:
    try:
        dfs.append(pd.read_csv(f))
    except Exception:
        dfs.append(pd.read_csv(f, sep='\t', header=None))

df_goemo = pd.concat(dfs, ignore_index=True)

print(f'\nOriginal shape: {df_goemo.shape}')
print(f'Columns: {list(df_goemo.columns)}')
df_goemo.head(3)

In [ ]:
# --- Trim GoEmotions ---
# GoEmotions has a 'labels' column (comma-separated emotion IDs)
# or one-hot columns per emotion — detect which format

emotion_col_goemo = None
for candidate in ['labels', 'label', 'emotion', 'emotions']:
    if candidate in df_goemo.columns:
        emotion_col_goemo = candidate
        break

print(f'Emotion column detected: {emotion_col_goemo}')

if emotion_col_goemo and df_goemo[emotion_col_goemo].nunique() > 1:
    # GoEmotions labels may be comma-separated — use first label for stratification
    df_goemo['_primary_label'] = df_goemo[emotion_col_goemo].astype(str).apply(
        lambda x: x.split(',')[0].strip()
    )
    n_classes = df_goemo['_primary_label'].nunique()
    per_class = TARGET_ROWS // n_classes
    print(f'Classes: {n_classes} | Rows per class: {per_class}')

    df_goemo_trimmed = (
        df_goemo.groupby('_primary_label', group_keys=False)
        .apply(lambda x: x.sample(min(len(x), per_class), random_state=RANDOM_SEED))
        .reset_index(drop=True)
        .drop(columns=['_primary_label'])
    )
else:
    df_goemo_trimmed = df_goemo.sample(n=min(TARGET_ROWS, len(df_goemo)), random_state=RANDOM_SEED).reset_index(drop=True)

print(f'\nOriginal rows : {len(df_goemo)}')
print(f'Trimmed rows  : {len(df_goemo_trimmed)}')
df_goemo_trimmed.head(3)

---
## ✅ Step 2 — Final Validation

In [ ]:
results = {
    'EmpatheticDialogues': df_emp_trimmed,
    'SpotifyLyrics'      : df_spot_trimmed,
    'GoEmotions'         : df_goemo_trimmed,
}

print('='*50)
print('FINAL SUMMARY')
print('='*50)
for name, df in results.items():
    status = '✅' if len(df) < 2000 else '❌ OVER LIMIT'
    print(f'{status}  {name}: {len(df):,} rows × {len(df.columns)} cols')
print('='*50)

---
## 💾 Step 3 — Save & Download CSVs

In [ ]:
output_dir = '/kaggle/working'

output_files = {
    'empathetic_dialogues_trimmed.csv': df_emp_trimmed,
    'spotify_lyrics_trimmed.csv'      : df_spot_trimmed,
    'goemotions_trimmed.csv'          : df_goemo_trimmed,
}

for filename, df in output_files.items():
    path = os.path.join(output_dir, filename)
    df.to_csv(path, index=False, encoding='utf-8')
    size_kb = os.path.getsize(path) / 1024
    print(f'Saved: {filename} ({len(df):,} rows, {size_kb:.1f} KB)')

print('\n🎉 All files saved to /kaggle/working — click the Output tab to download!')

In [ ]:
# Quick preview of each saved file
for filename in output_files:
    path = os.path.join(output_dir, filename)
    preview = pd.read_csv(path)
    print(f'\n--- {filename} ---')
    print(f'Shape: {preview.shape}')
    display(preview.head(2))